In [2]:
# Step 1: Set up the Sentinel-2 imagery time series for the AOI.
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import stackstac
from pystac_client import Client
from distributed.diagnostics import MemorySampler

from src.utils import (
    api_url,
    search_items,
    dedupe_items,
    stack_items,
    mask_clouds,
    ndwi,
    water_area,
    load_stations,
    stations_within,
    load_ghcn,
    daily_values,
    completeness,
    monthly_totals,
)

# The AOI is the Folsom Reservoir bounding box; use its centroid for station search.
bbox = (-121.18, 38.69, -121.04, 38.81)
aoi_lon = (bbox[0] + bbox[2]) / 2
aoi_lat = (bbox[1] + bbox[3]) / 2

# October 2018 is needed to complete water year 2019.
start = "2018-10-01"
end = "2025-12-31"

max_cloud = 40
mgrs_tile = "10SFH"

# search for items
items = list(search_items(bbox, start, end, max_cloud))

print("Scenes found:", len(items))

# remove the duplicats
items = dedupe_items(items, mgrs_tile)

print("Scenes after deduplication:", len(items))

# builtd the lazy stack
assets = ["green", "nir", "scl"]
stack = stack_items(
    items,
    assets,
    bbox,
    resolution=20,
    epsg=32610
)

# Keep the raw, filtered, deduplicated Part 1 stack for size/chunk accounting.
part1_stack = stackstac.stack(
    items,
    assets=["B03", "B08", "SCL"],
    resolution=20,
    epsg=32610,
    bounds_latlon=bbox,
    dtype="float32",
    fill_value=np.float32(np.nan),
    rescale=False,
)
print(part1_stack)



Scenes found: 800
Scenes after deduplication: 353
(<xarray.DataArray (time: 353, band: 2, y: 679, x: 623)> Size: 2GB
dask.array<truediv, shape=(353, 2, 679, 623), dtype=float64, chunksize=(1, 1, 679, 623), chunktype=numpy.ndarray>
Coordinates: (12/44)
  * time                                     (time) datetime64[ns] 3kB 2019-0...
    id                                       (time) <U54 76kB 'S2A_MSIL2A_201...
    platform                                 (time) <U11 16kB 'Sentinel-2A' ....
    s2:thin_cirrus_percentage                (time) float64 3kB 8.166 ... 0.3181
    s2:unclassified_percentage               (time) float64 3kB 14.35 ... 0.0...
    s2:nodata_pixel_percentage               (time) float64 3kB 3e-06 ... 0.0
    ...                                       ...
    instruments                              <U3 12B 'msi'
    proj:code                                <U10 40B 'EPSG:32610'
    sat:relative_orbit                       int64 8B 113
    s2:datatake_type           

In [ ]:
# Step 2: Find candidate weather stations within 30 km of the AOI centroid.
stations_url = "https://noaa-ghcn-pds.s3.amazonaws.com/ghcnd-stations.txt"
stations = load_stations(stations_url)
candidate_stations = stations_within(stations, aoi_lon, aoi_lat, radius_km=30)
display(candidate_stations[["station", "name", "latitude", "longitude", "distance_km"]])

# Step 3: Load PRCP for October 2018 through December 2025 and assess completeness.
ghcn_years = range(2018, 2026)
ghcn = load_ghcn("PRCP", ghcn_years)
candidate_ids = candidate_stations["station"].tolist()
candidate_daily = daily_values(ghcn, candidate_ids)
station_completeness = completeness(candidate_daily, "2018-10-01", "2025-12-31")
station_summary = candidate_stations.set_index("station").join(
    station_completeness.rename("completeness")
).sort_values("completeness", ascending=False)
display(station_summary[["name", "distance_km", "completeness"]])

# Choose the most complete nearby station; distance breaks ties through the prior sort.
selected_station = station_summary.index[0]
selected_row = station_summary.iloc[0]
print(
    f"Selected {selected_station} ({selected_row['name'].strip()}) because it has "
    f"the highest completeness ({selected_row['completeness']:.1%}) among stations "
    f"within 30 km of the AOI centroid."
)

# Step 4: Keep one station, then repartition and persist as in Lecture 14.
daily_station = (
    daily_values(ghcn, [selected_station])
    .repartition(npartitions=1)
    .persist()
)
prcp_monthly = monthly_totals(daily_station).rename("prcp_mm")

# Step 5: Reduce the lazy imagery to monthly median water area and scene counts.
reflectance_stack, scl_stack = stack
water_pixels = water_area(ndwi(reflectance_stack.where(~scl_stack.isin((0, 1, 3, 8, 9, 10)))))
water_observations = water_pixels[["water_km2"]].to_dataframe().reset_index()
water_observations["month"] = pd.to_datetime(water_observations["time"]).dt.to_period("M").dt.to_timestamp()
water_monthly = (
    water_observations.groupby("month")
    .agg(water_km2_median=("water_km2", "median"), n_scenes=("water_km2", "count"))
    .sort_index()
)

# Step 6: Join the records and make the requested two-panel shared-axis figure.
monthly = water_monthly.join(prcp_monthly, how="outer").sort_index()
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True, constrained_layout=True)
axes[0].bar(monthly.index, monthly["prcp_mm"], width=25, color="steelblue")
axes[0].set_ylabel("Precipitation (mm)")
axes[0].set_title(f"Monthly precipitation at {selected_station}")
axes[1].plot(monthly.index, monthly["water_km2_median"], marker="o", color="darkgreen")
axes[1].set_ylabel("Median water area (km²)")
axes[1].set_xlabel("Month")
axes[1].set_title("Monthly median Sentinel-2 water area")
plt.show()

# Step 7: Aggregate to water years (October 1 through September 30).
monthly = monthly.reset_index(names="month")
monthly["water_year"] = monthly["month"].dt.year + (monthly["month"].dt.month >= 10).astype(int)
water_year = (
    monthly.groupby("water_year")
    .agg(total_prcp_mm=("prcp_mm", "sum"), max_water_km2=("water_km2_median", "max"), n_months=("water_km2_median", "count"))
)
# Only retain water years with all twelve months represented by imagery.
water_year = water_year[water_year["n_months"] == 12]
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(water_year["total_prcp_mm"], water_year["max_water_km2"], color="darkorange")
for year, row in water_year.iterrows():
    ax.annotate(str(year), (row["total_prcp_mm"], row["max_water_km2"]), xytext=(5, 5), textcoords="offset points")
ax.set_xlabel("Water-year precipitation (mm)")
ax.set_ylabel("Maximum monthly water area (km²)")
ax.set_title("Water-year precipitation versus maximum reservoir area")
plt.show()

# Step 8: Export the required monthly table.
monthly_export = monthly[["month", "water_km2_median", "n_scenes", "prcp_mm"]]
os.makedirs("results", exist_ok=True)
monthly_export.to_csv("results/folsom_monthly.csv", index=False)
display(monthly_export.head())

# Step 9: Interpretation.
print(
    "Interpretation: precipitation and reservoir area may be positively related because "
    "wet water years can increase inflow and storage. The relationship is not expected "
    "to be one-to-one because reservoir operations, upstream runoff, evaporation, and "
    "seasonal lag also affect water area. A single station samples local precipitation, "
    "which may not represent rainfall across the entire watershed. In addition, snowpack "
    "and groundwater contributions can influence inflow independently of that station's "
    "daily precipitation."
)


In [ ]:
# Step 10: Build a naive, unfiltered stack for comparison without computing it.
naive_search = Client.open(api_url).search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime=f"{start}/{end}",
)
naive_items = list(naive_search.item_collection())
naive_stack = stackstac.stack(
    naive_items,
    assets=["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B10", "B11", "B12", "SCL"],
    resolution=10,
    epsg=32610,
    bounds_latlon=bbox,
    rescale=False,
)

def stack_metrics(array):
    """Summarize lazy stack size and Dask graph chunk count."""
    data = array.data
    return {
        "time_steps": array.sizes["time"],
        "size_GiB": data.nbytes / 1024**3,
        "chunks": int(np.prod(data.numblocks)),
    }

stack_comparison = pd.DataFrame(
    [stack_metrics(part1_stack), stack_metrics(naive_stack)],
    index=["Part 1", "Naive"],
)
display(stack_comparison)

# Step 11: Sample cluster memory while computing Part 1 water area and PRCP completeness.
water = water_pixels
ms = MemorySampler()
with ms.sample("water area"):
    water_result = water.compute()
with ms.sample("completeness"):
    completeness_result = completeness(candidate_daily, "2018-10-01", "2025-12-31")
ms.plot(align=True)
plt.title("Cluster memory during Part 1 and Part 2 computations")
plt.ylabel("Memory (bytes)")
plt.show()

# Step 12: Interpret the memory results relative to the 4 GiB budget.
print(
    "The largest stack reduction comes from the Part 1 choices that restrict the "
    "search to the AOI and one tile, remove duplicate dates, select three assets, "
    "and use 20 m resolution instead of the naive 10 m all-band stack. The peak "
    "cluster memory in the MemorySampler plot should be compared with the 4 GiB "
    "budget and is expected to exceed the lazy stack's reported size only by the "
    "temporary arrays and scheduler overhead needed during reduction. The same "
    "reduction succeeds on the naive stack because Dask processes it chunk by chunk "
    "and the computation only materializes reduced water-area results rather than "
    "the complete multi-band raster in memory."
)
